In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Bokeh plotting library and functions. Note, this is probably an overkill but I have used most of these in my other projects.
from bokeh.io import output_notebook, show
from bokeh.models.annotations.labels import Label
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Whisker, BoxAnnotation, Arrow, OpenHead, Span
from bokeh.plotting import figure, show, output_file, save
from bokeh.models import Legend, LinearAxis, Range1d, ColumnDataSource, LabelSet, HoverTool, DatetimeTickFormatter

from ecmwf.datastores import Client
import os
import time
import logging

import pvlib

In [2]:
current_dir = str(os.getcwd())

files_dir = current_dir + '/data/'

file_name = files_dir + 'cams_solar_rad_weather.csv'

In [ ]:
logging.basicConfig(level="INFO")

client = Client()
client.check_authentication()  # optional check

dataset = "cams-solar-radiation-timeseries"
# request = {
#     "sky_type": "observed_cloud",
#     "location": {"longitude": -5.68297, "latitude": 39.56428},
#     "altitude": ["-999."],
#     "date": ["2014-12-31/2018-12-31"],
#     "time_step": "1hour",
#     "time_reference": "universal_time",
#     "data_format": "csv"
# }

request = {
    "sky_type": "observed_cloud",
    "location": {"longitude": -6.26031, "latitude": 37.43123},
    "altitude": ["-999."],
    "date": ["2014-12-31/2018-12-31"],
    "time_step": "1minute",
    "time_reference": "universal_time",
    "data_format": "csv_expert"
}

remote_job = client.submit(dataset, request)

while not remote_job.results_ready:
    # Update the status information
    remote_job.update()
    
    # Show the current status
    print(f"Status: {remote_job.status}")
    
    # If the job is finished but had an error
    if remote_job.status == "failed":
        print("❌ The request failed.")
        break
        
    # Wait for 10 seconds before checking again
    print("Waiting 10 seconds before checking again...")
    time.sleep(10)

# Download the data if it's ready
if remote_job.results_ready:
    remote_job.download(target=file_name)
    print("✅ Download complete!")

In [ ]:
file_name = files_dir + 'cams_solar_rad_weather_example.csv'

def read_csv_with_header_last_comment(filepath, encoding="utf-8", comment_char="#", sep=";"):
    
    filepath = Path(filepath)

    # ---- pass 1: find the last comment line that contains the header ----
    last_header_line = None
    with filepath.open("r", encoding=encoding, newline="") as f:
        for line in f:
            s = line.strip()
            if s.startswith(comment_char):
                # remove leading "#" (and any following space)
                payload = s.lstrip(comment_char).strip()

                # skip empty comment lines like "#"
                if payload:
                    last_header_line = payload
            else:
                # first non-comment line -> comments are finished
                break

    if last_header_line is None:
        raise ValueError("No header found in the leading comment block.")

    # Split into column names
    colnames = [c.strip() for c in last_header_line.split(sep)]
    # Drop any empty column names (e.g., if line ends with ';')
    colnames = [c for c in colnames if c != ""]

    # ---- pass 2: read the data, skipping comment lines, using extracted header ----
    df = pd.read_csv(filepath, sep=sep, encoding=encoding, comment=comment_char, header=None, names=colnames)

    return df

# Example usage:
cams_seville_pd = read_csv_with_header_last_comment(file_name)
print(cams_seville_pd.head())
print(cams_seville_pd.tail())
print(cams_seville_pd.columns)

In [7]:
# Site location of solar power station near Seville
SITE_LAT  = 37.43123
SITE_LON  = -6.26031
NEAREST_CITY = "Seville"

# Seville (city) coordinates from Wikipedia
CITY_LAT = 37.39000
CITY_LON = -5.99000

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Vectorized haversine distance (km).
    lat/lon can be scalars or numpy arrays.
    """
    R = 6371.0088  # mean Earth radius in km
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# --- 1) Extract end time (after "/") and convert to datetime ---
end_time_str = cams_seville_pd["Observation period"].astype(str).str.split("/", n=1).str[1]

utc_dt = pd.to_datetime(end_time_str, errors="coerce").dt.floor("s")  # removes decimals

# Store utc times as strings in the format "YYYY-MM-DD HH:MM:SS":
# utc_time_str = utc_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

# --- 2) Build the new dataframe with selected columns ---
cols_keep = ["Clear sky GHI", "GHI", "Reliability", "Snow probability", "Cloud optical depth", "Cloud coverage", "Cloud type"]

cams_extra_seville_pd = cams_seville_pd.loc[:, cols_keep].copy()

# Put utc_time first
cams_extra_seville_pd.insert(0, "utc_time", utc_dt)

# --- 3) Add constant metadata columns ---
cams_extra_seville_pd["loc_lat"] = SITE_LAT
cams_extra_seville_pd["loc_long"] = SITE_LON
cams_extra_seville_pd["city_name"] = NEAREST_CITY

# --- 4) Distance to Seville city centre (km) ---
dist_km = haversine_km(SITE_LAT, SITE_LON, CITY_LAT, CITY_LON)
cams_extra_seville_pd["distance"] = dist_km

cams_extra_seville_pd.head()

,utc_time,Clear sky GHI,GHI,Reliability,Snow probability,Cloud optical depth,Cloud coverage,Cloud type,loc_lat,loc_long,city_name,distance
0,2014-12-31 00:01:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
1,2014-12-31 00:02:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
2,2014-12-31 00:03:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
3,2014-12-31 00:04:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
4,2014-12-31 00:05:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064


In [3]:
# Let's visualize power data by plotting power generation/demand for the various power sources along with
# some optional weather features.

# Set up plotting figure. Set x-axis to "datetime" so that the date time can be displayed appropriately.
def explore_plots(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black', color_features='black',
                  labels=None, features_labels=None, symbols=None, symbols_features=None, replaced_columns=None,
                  shade_daytime=False, daytime_column="daytime", daytime_alpha=0.15):

    """
    Function to create interactive exploratory plots.
    
    Parameters
    ----------
    dataframe : Pandas dataframe object
        The Pandas dataframe holding the data you want to plot.
    x_axis_column : string
        The name of the column you want to plot along the x-axis.
    y_axis_columns : list
        The names of the columns for the primary data you want to plot along the y-axis.
    x_axis_label : string
        The label to use for the x axis.
    y_axis_label : string
        The label to use for the y axis for the primary data. Note, all column data plotted will have the same y-axis scale
        and label name.
    title : string
        The title for the plot.

    Optional
    --------
    feature_columns : list or string
        The names of the columns you want to plot as additional features along secondary y-axis
        Default: None
    features_ylabel : list or string
        The secondary y-axis feature label/s. If a list greater than one item, list is concatenated into a single string.
        Default: None    
    p : bokeh plotting object
        A bokeh plotting object to add additional plotting objects to.
        Default: None
    normalize : Boolean
        Whether to normalize the data. Caution, might not work well for some features or when including more than one feature.
        Default: False
    other_colors : Boolean
        Whether to use your own colors (True) or to use color names as defined in this function (False).
        Default: False
    color_plot : string or list
        A string or list of colors to use for plotting the primary data. If list, must be length of number y_axis_columns.
        Default: 'black'
    color_features : string or list
        A string or list of colors to use for plotting the features data. If list, must be length of number feature_columns.
        Default: 'black'
    labels : list or string
        The labels to assign all the primary data from the y_axis_columns, should be list of length y_axis_columns if more than one
        primary data is being plotted.
        Default: None
    features_labels : list or string
        The labels to assign all the features data from feature_columns, should be list of length feature_columns if more than one
        feature is being plotted.
        Default: None
    symbols : string or list
        The plotting markers for the primary data. If list, must be length of number of y_axis_columns.
        Default: None
    symbols_features : string or list
        The plotting markers for the features data. If list, must be length of number of feature_columns.
        Default: None
    replaced_columns : Boolean
        Whether to plot the replaced primary data produced through cleaning. The appropriate columns must exist in the dataframe
        if True and given as 'replaced_' and the y_axis_columns, e.g., 'replaced_energy_usage_Wh' for replacements for 
        the 'energy_usage_Wh'.
        Default: False
    shade_daytime : Boolean
        Plot a transparent box around the daytime (when the Sun is above the horizon) if True. Requires a 'daytime' column in the
        input dataframe.
        Default: False
    daytime_column : string
        The column name containing the booleans on whether it is daytime or not.
        Default: "daytime"
    daytime_alpha : float
        The alpha value to use for creating the filled in box during daytime.
        Default: 0.15

    Returns
    -------
    p : bokeh object
        The Bokeh plotting object.
    """
    
    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}

    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        # Rotate labels for better readability
        p.xaxis.major_label_orientation = 120
        # Reduce the number of x-axis ticks to avoid crowding
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

        # ---- NEW: shade daytime regions (underlay) ----
        if shade_daytime and (daytime_column in dataframe.columns):
            df_day = dataframe[[x_axis_column, daytime_column]].dropna().sort_values(x_axis_column)
            if not df_day.empty:
                x = df_day[x_axis_column].values
                d = df_day[daytime_column].astype(bool).values

                # Find contiguous True segments
                in_seg = False
                seg_start = None
                for i in range(len(d)):
                    if d[i] and not in_seg:
                        in_seg = True
                        seg_start = x[i]
                    # segment ends when it flips False OR at last point
                    if in_seg and ((not d[i]) or (i == len(d) - 1)):
                        seg_end = x[i] if (i == len(d) - 1 and d[i]) else x[i-1]
                        box = BoxAnnotation(
                            left=seg_start, right=seg_end,
                            fill_color=colors["yellow"], fill_alpha=daytime_alpha,
                            line_alpha=0
                        )
                        p.add_layout(box)
                        in_seg = False
                        seg_start = None

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    # print('labels: ', labels)
    # print('markers: ', symbols)
    # print('colors: ', color_plot_values)
    
    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        # Create a loop to plot the column data for each city in Spain
        for y, city in enumerate(dataframe['city_name'].unique()):

            if normalize:
                max_primary_data = dataframe.loc[dataframe['city_name'] == city, column].max()
            else:
                max_primary_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 | y == 0 else False
            
            # Scatter plot with symbols.
            if symbols:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data, size=10,
                          marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                          legend_label=label + ' ' + str(city), visible=visible)
                
            # Add a line to connect the symbols
            p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                   dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data,
                   line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label + ' ' + str(city),
                   visible=visible)

            if replaced_columns:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, f'replaced_{column}']/max_primary_data,
                          size=10, color="red", marker="diamond", alpha=0.5, legend_label="Replaced " + label + ' ' + str(city),
                          visible=visible)

            # print('marker_index: ', marker_index)
            # print('color_index: ', color_index)
            # print('label_index: ', label_index)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
        label_index = label_index + 1
        

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        if normalize:
            max_features_data = dataframe[feature_columns].max().max()
        else:
            max_features_data = 1

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            # Create a loop to plot the features for each city.
            for y, city in enumerate(dataframe['city_name'].unique()):

                if normalize:
                    max_features_data = dataframe.loc[dataframe['city_name'] == city, feature_column].max()
                else:
                    max_features_data = 1

                # Only show the first column by default, hide others
                visible = True if i == 0 | y == 0 else False
                
                # Scatter plot with symbols. Only plot the energy usage as a function of time for a specific city.
                if symbols_features:
                    p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                              dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data,
                              y_range_name="features", size=10, marker=symbols_features[marker_index],
                              color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label + ' ' + str(city),
                              visible=visible)
                # Add a line to connect the symbols
                p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                       dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data, y_range_name="features",
                       line_width=2, color=color_plot_values[color_index], alpha=0.5,
                       legend_label=feature_label + ' ' + str(city), visible=visible)
            
                marker_index = marker_index + 1
                color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis     
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    p.add_layout(Legend(), 'right')
    p.legend.click_policy="hide"
    p.legend.background_fill_alpha = 0.3
    p.legend.border_line_alpha = 0.2

    return(p)

In [11]:
p = explore_plots(cams_extra_seville_pd, 'utc_time', ["Clear sky GHI", "GHI"],
                  r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Irradiation\ (Wh/m)}$$",
                  'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time',
                  feature_columns=["Cloud optical depth", "Cloud coverage"],
                  features_ylabel=[r"$$\mathrm{Cloud\ optical\ depth\ and\ coverage}$$"], p=None, normalize=False, other_colors=False,
                  color_plot=['red', 'dorange'],
                  color_features=['blue', 'black'],
                  labels=['Clear Sky GHI', 'GHI with Clouds'], features_labels=['Cloud Optical Depth', 'Cloud Coverage'],
                  symbols=['star', 'triangle'],
                  symbols_features=['circle', 'square'])

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Ground_irradiation_vs_clouds.html'

title = 'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time'

save(p, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_31787/3500979379.py:25: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/exploratory/Ground_irradiation_vs_clouds.html'

In [ ]:
file_name = files_dir + 'cams_solar_rad_weather.csv'

def read_csv_with_header_last_comment(filepath, encoding="utf-8", comment_char="#", sep=";"):
    
    filepath = Path(filepath)

    # ---- pass 1: find the last comment line that contains the header ----
    last_header_line = None
    with filepath.open("r", encoding=encoding, newline="") as f:
        for line in f:
            s = line.strip()
            if s.startswith(comment_char):
                # remove leading "#" (and any following space)
                payload = s.lstrip(comment_char).strip()

                # skip empty comment lines like "#"
                if payload:
                    last_header_line = payload
            else:
                # first non-comment line -> comments are finished
                break

    if last_header_line is None:
        raise ValueError("No header found in the leading comment block.")

    # Split into column names
    colnames = [c.strip() for c in last_header_line.split(sep)]
    # Drop any empty column names (e.g., if line ends with ';')
    colnames = [c for c in colnames if c != ""]

    # ---- pass 2: read the data, skipping comment lines, using extracted header ----
    df = pd.read_csv(filepath, sep=sep, encoding=encoding, comment=comment_char, header=None, names=colnames)

    return df

# Example usage:
cams_seville_pd = read_csv_with_header_last_comment(file_name)
print(cams_seville_pd.head())
print(cams_seville_pd.tail())
print(cams_seville_pd.columns)

In [5]:
# Site location of solar power station near Seville
SITE_LAT  = 37.43123
SITE_LON  = -6.26031
NEAREST_CITY = "Seville"

# Seville (city) coordinates from Wikipedia
CITY_LAT = 37.39000
CITY_LON = -5.99000

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Vectorized haversine distance (km).
    lat/lon can be scalars or numpy arrays.
    """
    R = 6371.0088  # mean Earth radius in km
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# --- 1) Extract end time (after "/") and convert to datetime ---
end_time_str = cams_seville_pd["Observation period"].astype(str).str.split("/", n=1).str[1]

utc_dt = pd.to_datetime(end_time_str, errors="coerce").dt.floor("s")  # removes decimals

# Store utc times as strings in the format "YYYY-MM-DD HH:MM:SS":
# utc_time_str = utc_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

# --- 2) Build the new dataframe with selected columns ---
cols_keep = ["Clear sky GHI", "GHI", "Reliability", "Snow probability", "Cloud optical depth", "Cloud coverage", "Cloud type"]

cams_extra_seville_pd = cams_seville_pd.loc[:, cols_keep].copy()

# Put utc_time first
cams_extra_seville_pd.insert(0, "utc_time", utc_dt)

# --- 3) Add constant metadata columns ---
cams_extra_seville_pd["loc_lat"] = SITE_LAT
cams_extra_seville_pd["loc_long"] = SITE_LON
cams_extra_seville_pd["city_name"] = NEAREST_CITY

# --- 4) Distance to Seville city centre (km) ---
dist_km = haversine_km(SITE_LAT, SITE_LON, CITY_LAT, CITY_LON)
cams_extra_seville_pd["distance"] = dist_km

cams_extra_seville_pd.head()
cams_extra_seville_pd.tail()

,utc_time,Clear sky GHI,GHI,Reliability,Snow probability,Cloud optical depth,Cloud coverage,Cloud type,loc_lat,loc_long,city_name,distance
2105275,2018-12-31 23:56:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
2105276,2018-12-31 23:57:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
2105277,2018-12-31 23:58:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
2105278,2018-12-31 23:59:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
2105279,2019-01-01 00:00:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064


In [6]:
start_date = '2014-12-31'
end_date = '2015-07-01'
cams_extra_seville_pd_short = cams_extra_seville_pd[(cams_extra_seville_pd['utc_time'] >= start_date) & \
                              (cams_extra_seville_pd['utc_time'] <= end_date)].copy()

In [10]:
p = explore_plots(cams_extra_seville_pd_short, 'utc_time', ["Clear sky GHI", "GHI"],
                  r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Irradiation\ (Wh/m)}$$",
                  'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time',
                  feature_columns=["Cloud optical depth", "Cloud coverage"],
                  features_ylabel=[r"$$\mathrm{Cloud\ optical\ depth\ and\ coverage}$$"], p=None, normalize=True, other_colors=False,
                  color_plot=['red', 'dorange'],
                  color_features=['blue', 'black'],
                  labels=['Clear Sky GHI', 'GHI with Clouds'], features_labels=['Cloud Optical Depth', 'Cloud Coverage'],
                  symbols=['star', 'triangle'],
                  symbols_features=['circle', 'square'])

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Ground_irradiation_vs_clouds_all_obs.html'

title = 'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time'

save(p, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_71132/3198004691.py:25: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/exploratory/Ground_irradiation_vs_clouds_all_obs.html'

In [8]:
lat = float(cams_extra_seville_pd_short["loc_lat"].iloc[0])
lon = float(cams_extra_seville_pd_short["loc_long"].iloc[0])

# Calculate solar position
times = cams_extra_seville_pd_short["utc_time"]
solpos = pvlib.solarposition.get_solarposition(time=times, latitude=lat, longitude=lon)

# Sun above horizon: apparent elevation > 0 deg
cams_extra_seville_pd_short["daytime"] = (solpos["apparent_elevation"].to_numpy() > 0)

In [22]:
p = explore_plots(cams_extra_seville_pd_short, 'utc_time', ["Clear sky GHI", "GHI"],
                  r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Irradiation\ (Wh/m)}$$",
                  'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time',
                  feature_columns=["Cloud optical depth", "Cloud coverage"],
                  features_ylabel=[r"$$\mathrm{Cloud\ optical\ depth\ and\ coverage}$$"], p=None, normalize=True, other_colors=False,
                  color_plot=['red', 'dorange'],
                  color_features=['blue', 'black'],
                  labels=['Clear Sky GHI', 'GHI with Clouds'], features_labels=['Cloud Optical Depth', 'Cloud Coverage'],
                  symbols=['star', 'triangle'],
                  symbols_features=['circle', 'square'],
                  shade_daytime=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Ground_irradiation_vs_clouds_all_obs2.html'

title = 'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time'

save(p, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_59349/3011229126.py:26: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/exploratory/Ground_irradiation_vs_clouds_all_obs2.html'

In [14]:
def fill_cams_data_day_night_rules(
    df: pd.DataFrame,
    time_col: str = "utc_time",
    daytime_col: str = "daytime",
    taper: str = "30min",
    night_value: float = 0.0,
    cont_cols=("Cloud coverage", "Cloud optical depth", "Snow probability"),
    cloud_type_col: str = "Cloud type",
    invalid_sentinels=(-1,),
    clip_specs=None,
    use_day_rolling_mean: bool = True,
    rolling_window: str = "1h",
    rolling_min_periods: int = 1,
) -> pd.DataFrame:

    if clip_specs is None:
        clip_specs = {
            cloud_type_col: (0, 8),
            "Cloud coverage": (0, 100),
            "Snow probability": (0, 100),
        }

    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce")

    # Recommended: drop any rows where time couldn't be parsed
    out = out.dropna(subset=[time_col]).sort_values(time_col)

    dfi = out.set_index(time_col)

    if daytime_col not in dfi.columns:
        raise ValueError(f"'{daytime_col}' column not found. Create it before calling this function.")

    day = dfi[daytime_col].astype(bool)
    night = ~day
    idx = dfi.index

    def is_invalid(series):
        m = series.isna()
        for s in invalid_sentinels:
            m |= (series == s)
        return m

    cols_all = list(cont_cols) + [cloud_type_col]
    for col in cols_all:
        if col in dfi.columns:
            dfi[col] = pd.to_numeric(dfi[col], errors="coerce").astype("Float64")

    # 1) Night rule: invalid -> night_value
    for col in cols_all:
        if col not in dfi.columns:
            continue
        invalid = is_invalid(dfi[col])
        dfi.loc[night & invalid, col] = night_value

    # 2) Optional daytime rolling mean fill for continuous columns (daytime-only contributors)
    if use_day_rolling_mean:
        for col in cont_cols:
            if col not in dfi.columns:
                continue
            invalid = is_invalid(dfi[col])
            mask = day & invalid

            s = dfi[col].where(day, np.nan)
            s = s.mask(day & invalid, np.nan)
            smooth = s.rolling(rolling_window, center=True, min_periods=rolling_min_periods).mean()

            dfi.loc[mask, col] = smooth.loc[mask]

    # 3) Taper weights for sunrise/sunset blending (POSITIONAL: no index alignment issues)
    taper_seconds = pd.Timedelta(taper).total_seconds()

    day_np = day.to_numpy(dtype=bool)
    day_int = day_np.astype(np.int8)

    boundary = np.empty_like(day_int, dtype=bool)
    boundary[0] = False
    boundary[1:] = day_int[1:] != day_int[:-1]

    times_ns = idx.view("i8")
    boundary_ns = np.where(boundary, times_ns, np.nan)

    last_boundary_ns = pd.Series(boundary_ns).ffill().to_numpy()
    next_boundary_ns = pd.Series(boundary_ns).bfill().to_numpy()

    # Fill any remaining NaNs (e.g., if no boundary exists in the slice)
    last_boundary_ns = pd.Series(last_boundary_ns).fillna(times_ns[0]).to_numpy()
    next_boundary_ns = pd.Series(next_boundary_ns).fillna(times_ns[-1]).to_numpy()

    dt_since = (times_ns - last_boundary_ns) / 1e9
    dt_until = (next_boundary_ns - times_ns) / 1e9
    dist_to_boundary = np.minimum(dt_since, dt_until)

    # print("rows:", len(dfi), "index unique:", dfi.index.is_unique, "NaT times:", dfi.index.isna().sum())

    w = np.clip(dist_to_boundary / taper_seconds, 0.0, 1.0)
    w = np.where(day_np, w, 0.0)

    # 4) Remaining daytime invalids for continuous cols: daytime interpolation + taper blend
    for col in cont_cols:
        if col not in dfi.columns:
            continue

        invalid = is_invalid(dfi[col])
        mask = day & invalid

        if mask.any():
            s_day = dfi[col].where(day, np.nan)
            s_day = s_day.interpolate(method="time", limit_direction="both")

            blended = (w * s_day.to_numpy()) + ((1.0 - w) * night_value)
            dfi.loc[mask, col] = pd.Series(blended, index=idx).loc[mask]

    # 5) Cloud type: remaining daytime invalids -> nearest valid daytime value (POSITIONAL)
    if cloud_type_col in dfi.columns:
        invalid = is_invalid(dfi[cloud_type_col])
        mask = day & invalid
    
        if mask.any():
            s = dfi[cloud_type_col].to_numpy(dtype="float64")   # may contain nan
            day_np = day.to_numpy(dtype=bool)
    
            # valid daytime points for Cloud type
            valid = day_np & np.isfinite(s)
    
            # forward nearest: last valid index at or before i
            prev_idx = np.full(len(s), -1, dtype=int)
            last = -1
            for i in range(len(s)):
                if valid[i]:
                    last = i
                prev_idx[i] = last
    
            # backward nearest: next valid index at or after i
            next_idx = np.full(len(s), -1, dtype=int)
            nxt = -1
            for i in range(len(s) - 1, -1, -1):
                if valid[i]:
                    nxt = i
                next_idx[i] = nxt
    
            # choose whichever is closer in time; if one side missing, use the other
            choose = np.zeros(len(s), dtype=bool)  # True -> use prev
            for i in range(len(s)):
                p = prev_idx[i]
                n = next_idx[i]
                if p == -1 and n == -1:
                    choose[i] = True  # doesn't matter; will stay nan
                elif p == -1:
                    choose[i] = False
                elif n == -1:
                    choose[i] = True
                else:
                    choose[i] = (i - p) <= (n - i)
    
            filled = s.copy()
            use_prev = choose
            use_next = ~choose
    
            # apply fill only where needed (daytime invalids)
            m = mask.to_numpy(dtype=bool)
            # fill from prev or next indices
            filled[m & use_prev & (prev_idx != -1)] = s[prev_idx[m & use_prev & (prev_idx != -1)]]
            filled[m & use_next & (next_idx != -1)] = s[next_idx[m & use_next & (next_idx != -1)]]
    
            dfi.loc[mask, cloud_type_col] = filled[m]

    # 6) Round/clip/cast integer-coded cols; keep optical depth float
    for col, (vmin, vmax) in clip_specs.items():
        if col in dfi.columns:
            dfi[col] = dfi[col].round().clip(vmin, vmax).astype("Int64")

    if "Cloud optical depth" in dfi.columns:
        dfi["Cloud optical depth"] = dfi["Cloud optical depth"].astype("Float64")

    return dfi.reset_index()

In [15]:
cams_extra_seville_pd_short = fill_cams_data_day_night_rules(
    cams_extra_seville_pd_short,
    taper="30min",
    use_day_rolling_mean=True,
    rolling_window="30min",   # lowercase h to avoid warnings
)

rows: 262080 index unique: True NaT times: 0


In [33]:
p = explore_plots(cams_extra_seville_pd_short, 'utc_time', ["Clear sky GHI", "GHI"],
                  r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Irradiation\ (Wh/m^{2})}$$",
                  'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time',
                  feature_columns=["Cloud optical depth", "Cloud coverage", "Cloud type"],
                  features_ylabel=[r"$$\mathrm{Cloud\ optical\ depth,\ coverage,\ and\ type}$$"], p=None, normalize=False, 
                  other_colors=False, color_plot=['red', 'dorange'],
                  color_features=['blue', 'black', 'green'],
                  labels=['Clear Sky GHI', 'GHI with Clouds'], features_labels=['Cloud Optical Depth', 'Cloud Coverage', 'Cloud Type'],
                  symbols=['star', 'triangle'],
                  symbols_features=['circle', 'square', 'diamond'],
                  shade_daytime=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Ground_irradiation_vs_clouds_all_obs5.html'

title = 'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time'

save(p, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_39656/2391607744.py:26: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/exploratory/Ground_irradiation_vs_clouds_all_obs5.html'

In [18]:
# Count the null values in each column
null_counts = cams_extra_seville_pd_short.isnull().sum()

# Print the result
print(null_counts)

utc_time               0
Clear sky GHI          0
GHI                    0
Reliability            0
Snow probability       0
Cloud optical depth    0
Cloud coverage         0
Cloud type             0
loc_lat                0
loc_long               0
city_name              0
distance               0
daytime                0
dtype: int64


In [19]:
count_minus_one = (cams_extra_seville_pd_short == -1).sum()
print(count_minus_one)

utc_time               0
Clear sky GHI          0
GHI                    0
Reliability            0
Snow probability       0
Cloud optical depth    0
Cloud coverage         0
Cloud type             0
loc_lat                0
loc_long               0
city_name              0
distance               0
daytime                0
dtype: Int64


In [28]:
def resample_cams_to_hourly(df, time_col="utc_time"):
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
    d = d.dropna(subset=[time_col]).sort_values(time_col).set_index(time_col)

    def mode_or_na(s):
        s = s.dropna()
        return s.mode().iloc[0] if not s.empty else pd.NA

    agg = {
        # Irradiation (Wh/m^2 over 1 min) -> hourly energy (Wh/m^2 over 1 h)
        "Clear sky GHI": "sum",
        "GHI": "sum",

        # Clouds/snow: hourly mean conditions (reasonable default)
        "Cloud optical depth": "mean",
        "Cloud coverage": "mean",
        "Snow probability": "mean",
        "Cloud type": mode_or_na,

        # daytime: fraction of the hour that is daytime
        "daytime": "mean",

        # location metadata
        "loc_lat": "first",
        "loc_long": "first",
        "city_name": "first",
        "distance": "first",
    }

    # keep only existing columns
    agg = {k: v for k, v in agg.items() if k in d.columns}

    hourly = (
        d.resample("1h", closed="right", label="right")
         .agg(agg)
         .reset_index()
         .rename(columns={time_col: "utc_time"})
    )

    if "daytime" in hourly.columns:
        hourly["daytime_frac"] = hourly["daytime"]
        hourly["daytime_any"] = hourly["daytime_frac"] > 0

    return hourly

In [29]:
cams_extra_seville_pd_short_hourly = resample_cams_to_hourly(cams_extra_seville_pd_short)

In [34]:
p = explore_plots(cams_extra_seville_pd_short_hourly, 'utc_time', ["Clear sky GHI", "GHI"],
                  r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Irradiation\ (Wh/m^{2})}$$",
                  'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time',
                  feature_columns=["Cloud optical depth", "Cloud coverage", "Cloud type"],
                  features_ylabel=[r"$$\mathrm{Cloud\ optical\ depth,\ coverage,\ and\ type}$$"], p=None, normalize=True, 
                  other_colors=False, color_plot=['red', 'dorange'],
                  color_features=['blue', 'black', 'green'],
                  labels=['Clear Sky GHI', 'GHI with Clouds'], features_labels=['Cloud Optical Depth', 'Cloud Coverage', 'Cloud Type'],
                  symbols=['star', 'triangle'],
                  symbols_features=['circle', 'square', 'diamond'],
                  shade_daytime=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Ground_irradiation_vs_clouds_all_obs_hourly.html'

title = 'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time'

save(p, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_39656/691203157.py:26: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/exploratory/Ground_irradiation_vs_clouds_all_obs_hourly.html'